# Phase 20 — Multi-task Learning: BERT-base + Focal Loss

Ένα μοντέλο με shared BERT encoder και δύο classification heads.

**Αρχιτεκτονική:**
```
Text → BERT Encoder → [CLS] representation
                    → Hazard Head → hazard prediction
                    → Product Head → product prediction
```

**Loss:**
```
total_loss = focal_loss_hazard + focal_loss_product
```

**Γιατί Multi-task:**
- Ο shared encoder μαθαίνει representations χρήσιμες και για τα δύο tasks
- Εκμεταλλεύεται τη συσχέτιση hazard↔product (π.χ. biological → meat/dairy)
- Λιγότερες παράμετροι από δύο ξεχωριστά μοντέλα

**Τρέχον best:** 0.80548

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import random
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

train_full = pd.concat([train, valid], ignore_index=True)

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_full = make_text(train_full)
texts_test = make_text(test)

print(f'Train+Valid: {len(texts_full)}')
print(f'Test:        {len(texts_test)}')

In [ ]:
le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

y_full_hazard  = le_hazard.transform(train_full['hazard-category'])
y_full_product = le_product.transform(train_full['product-category'])

n_hazard  = len(le_hazard.classes_)
n_product = len(le_product.classes_)
print(f'Hazard classes:  {n_hazard}')
print(f'Product classes: {n_product}')

In [ ]:
# Focal Loss — χωρίς class weights (αυτό έδωσε 0.804)
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits, targets):
        ce_loss = F.cross_entropy(logits, targets, reduction='none')
        pt      = torch.exp(-ce_loss)
        focal   = (1 - pt) ** self.gamma * ce_loss
        return focal.mean()

criterion = FocalLoss(gamma=2.0)
print('Focal Loss initialized')


# Multi-task Model: shared BERT encoder + δύο classification heads
class MultiTaskBERT(nn.Module):
    def __init__(self, model_name, n_hazard, n_product, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size  = self.encoder.config.hidden_size

        # Δύο ξεχωριστά classification heads
        self.hazard_head  = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, n_hazard)
        )
        self.product_head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, n_product)
        )

    def forward(self, input_ids, attention_mask):
        # Shared encoder — παίρνουμε το [CLS] token
        outputs    = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token

        hazard_logits  = self.hazard_head(cls_output)
        product_logits = self.product_head(cls_output)
        return hazard_logits, product_logits

print('MultiTaskBERT architecture defined')

In [ ]:
MODEL_NAME   = 'bert-base-uncased'
BATCH_SIZE   = 32
MAX_LENGTH   = 128
LR           = 2e-5
N_EPOCHS     = 20  #tried 20, 25, 30, 35, 40
WARMUP_RATIO = 0.1

tokenizer    = AutoTokenizer.from_pretrained(MODEL_NAME)
dummy_labels = np.zeros(len(texts_test), dtype=int)
print(f'Tokenizer loaded: {MODEL_NAME}')

In [ ]:
# Dataset — επιστρέφει ΚΑΙ hazard ΚΑΙ product label
class MultiTaskDataset(Dataset):
    def __init__(self, texts, hazard_labels, product_labels, tokenizer, max_length=128):
        self.texts          = texts
        self.hazard_labels  = hazard_labels
        self.product_labels = product_labels
        self.tokenizer      = tokenizer
        self.max_length     = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoded['input_ids'].squeeze(),
            'attention_mask': encoded['attention_mask'].squeeze(),
            'hazard_label':   torch.tensor(self.hazard_labels[idx],  dtype=torch.long),
            'product_label':  torch.tensor(self.product_labels[idx], dtype=torch.long)
        }


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0
    for batch in loader:
        hazard_labels  = batch.pop('hazard_label').to(device)
        product_labels = batch.pop('product_label').to(device)
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()
        hazard_logits, product_logits = model(**batch)

        # Combined Focal Loss για hazard + product
        loss_hazard  = criterion(hazard_logits,  hazard_labels)
        loss_product = criterion(product_logits, product_labels)
        loss = loss_hazard + loss_product

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def get_predictions(model, loader):
    model.eval()
    all_hazard  = []
    all_product = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('hazard_label',  None)
            batch.pop('product_label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            hazard_logits, product_logits = model(**batch)
            all_hazard.extend(hazard_logits.argmax(dim=1).cpu().numpy())
            all_product.extend(product_logits.argmax(dim=1).cpu().numpy())
    return np.array(all_hazard), np.array(all_product)


def get_probabilities(model, loader):
    model.eval()
    all_hazard_probs  = []
    all_product_probs = []
    with torch.no_grad():
        for batch in loader:
            batch.pop('hazard_label',  None)
            batch.pop('product_label', None)
            batch = {k: v.to(device) for k, v in batch.items()}
            hazard_logits, product_logits = model(**batch)
            all_hazard_probs.append(torch.softmax(hazard_logits,  dim=1).cpu().numpy())
            all_product_probs.append(torch.softmax(product_logits, dim=1).cpu().numpy())
    return np.vstack(all_hazard_probs), np.vstack(all_product_probs)


def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product, verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard, average='macro', zero_division=0)
    mask      = (np.array(y_true_hazard) == np.array(y_pred_hazard))
    f1_product = f1_score(
        np.array(y_true_product)[mask],
        np.array(y_pred_product)[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_hazard + f1_product) / 2
    if verbose:
        print(f'macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'Σωστά hazard predictions:           {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

In [ ]:
# DataLoaders — ένα dataset με ΚΑΙ τα δύο labels
full_loader = DataLoader(
    MultiTaskDataset(texts_full, y_full_hazard, y_full_product, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=True
)
test_loader = DataLoader(
    MultiTaskDataset(texts_test, dummy_labels, dummy_labels, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=False
)

print(f'Train batches: {len(full_loader)}')
print(f'Batch size: {BATCH_SIZE} | Epochs: {N_EPOCHS} | LR: {LR}')

## Εκπαίδευση Multi-task BERT

In [ ]:
print('=== ΕΚΠΑΙΔΕΥΣΗ Multi-task BERT-base + Focal Loss ===')
print(f'Δείγματα: {len(texts_full)} | Epochs: {N_EPOCHS} | LR: {LR}\n')

model = MultiTaskBERT(MODEL_NAME, n_hazard, n_product).to(device)

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(full_loader) * N_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
print(f'Total steps: {total_steps} | Warmup steps: {warmup_steps}\n')
print(f'Παράμετροι μοντέλου: {sum(p.numel() for p in model.parameters()):,}\n')

for epoch in range(N_EPOCHS):
    loss = train_epoch(model, full_loader, optimizer, scheduler)
    print(f'Epoch {epoch+1}/{N_EPOCHS} | Combined Focal Loss: {loss:.4f}')

print('\nΕκπαίδευση ολοκληρώθηκε')

## Αποθήκευση Probabilities + Submission

In [ ]:
# Test probabilities για ensemble
test_hazard_probs, test_product_probs = get_probabilities(model, test_loader)

np.save('multitask_test_hazard_probs.npy',  test_hazard_probs)
np.save('multitask_test_product_probs.npy', test_product_probs)
print('Αποθηκεύτηκαν τα .npy')

# Predictions
test_hazard_preds, test_product_preds = get_predictions(model, test_loader)
test_hazard  = le_hazard.inverse_transform(test_hazard_preds)
test_product = le_product.inverse_transform(test_product_preds)

pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_hazard,
    'product-category': test_product
}).to_csv('submission_multitask.csv', index=False)

print('Αποθηκεύτηκε: submission_multitask.csv')